Basic Install

In [ ]:
!pip -q install -U pyserini faiss-cpu sentence-transformers huggingface_hub
!apt-get -qq install -y openjdk-21-jdk-headless > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


Download Hindi Corpus

In [ ]:
import gzip, json
from huggingface_hub import HfApi, hf_hub_download

WORK = "/content/miracl_hi"
os.makedirs(WORK, exist_ok=True)

# find all Hindi shards automatically
files = [f for f in HfApi().list_repo_files("miracl/miracl-corpus", repo_type="dataset")
         if f.startswith("miracl-corpus-v1.0-hi/") and f.endswith(".jsonl.gz")]
print(files)

shard_paths = [hf_hub_download("miracl/miracl-corpus", f, repo_type="dataset") for f in sorted(files)]

def iter_passages():
    for p in shard_paths:
        with gzip.open(p, "rt", encoding="utf-8") as f:
            for line in f:
                d = json.loads(line)
                yield d["docid"], d["title"], d["text"]

N = sum(1 for _ in iter_passages())
print(N, "passages")

['miracl-corpus-v1.0-hi/docs-0.jsonl.gz', 'miracl-corpus-v1.0-hi/docs-1.jsonl.gz']


miracl-corpus-v1.0-hi/docs-0.jsonl.gz: reconstructing file:   0%|          |  0.00B / 95.7MB            

miracl-corpus-v1.0-hi/docs-0.jsonl.gz: downloading bytes:           |  0.00B            

miracl-corpus-v1.0-hi/docs-1.jsonl.gz: reconstructing file:   0%|          |  0.00B / 1.16MB            

miracl-corpus-v1.0-hi/docs-1.jsonl.gz: downloading bytes:           |  0.00B            

506264 passages


Index 1, BM25 with the Hindi analyzer

In [ ]:
bm25_in = f"{WORK}/bm25_input"
os.makedirs(bm25_in, exist_ok=True)
with open(f"{bm25_in}/docs.jsonl", "w", encoding="utf-8") as out:
    for docid, title, text in iter_passages():
        out.write(json.dumps({"id": docid, "contents": f"{title} {text}"}, ensure_ascii=False) + "\n")

!python -m pyserini.index.lucene \
  --collection JsonCollection \
  --input {bm25_in} \
  --index {WORK}/bm25_index \
  --generator DefaultLuceneDocumentGenerator \
  --language hi \
  --threads 4

[0.015s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.015s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-10-25 12:21:00,751 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:208) - Setting log level to INFO
2026-10-25 12:21:00,755 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:211) - ============ Loading Index Configuration ============
2026-10-25 12:21:00,756 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:212) - AbstractIndexer settings:
2026-10-25 12:21:00,757 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:213) -  + DocumentCollection path: /content/miracl_hi/bm25_input
2026-10-25 12:21:00,757 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:214) -  + CollectionClass: JsonCollection
2026-10-25 12:21:00,758 INFO  [m

Test it

In [ ]:
from pyserini.search.lucene import LuceneSearcher

bm25 = LuceneSearcher(f"{WORK}/bm25_index")
bm25.set_language("hi")          # same analyzer as at indexing time
bm25.set_bm25(k1=0.9, b=0.4)

q = "पेनिसिलिन की खोज किसने की"
hits = bm25.search(q, 3)
print(bm25.num_docs, [(h.docid, round(h.score, 2)) for h in hits])

506264 [('35196#5', 9.56), ('47306#3', 9.56), ('47314#6', 9.55)]


Index 2

In [ ]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")
model.half()
model.max_seq_length = 256
DIM = model.get_sentence_embedding_dimension()

emb = np.lib.format.open_memmap(f"{WORK}/emb.npy", mode="w+", dtype="float16", shape=(N, DIM))
docids = []
BATCH, CHUNK = 128, 20_000
buf_txt, buf_ids, pos = [], [], 0

def flush():
    global pos, buf_txt, buf_ids
    if not buf_txt: return
    v = model.encode(buf_txt, batch_size=BATCH, normalize_embeddings=True,
                     convert_to_numpy=True, show_progress_bar=False)
    emb[pos:pos+len(v)] = v
    docids.extend(buf_ids)
    pos += len(v)
    print(f"{pos}/{N}", end="\r")
    buf_txt, buf_ids = [], []

for docid, title, text in iter_passages():
    buf_txt.append(f"passage: {title} {text}")
    buf_ids.append(docid)
    if len(buf_txt) == CHUNK: flush()
flush()
emb.flush()

json.dump(docids, open(f"{WORK}/docids.json", "w"))

index = faiss.IndexFlatIP(DIM)
for s in range(0, N, 100_000):
    index.add(np.asarray(emb[s:s+100_000], dtype="float32"))
faiss.write_index(index, f"{WORK}/dense.faiss")
print("\nDense index size:", index.ntotal)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

/tmp/ipykernel_6580/2103233957.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = model.get_sentence_embedding_dimension()



Dense index size: 506264


Test it

In [ ]:
docid2text = {d: f"{t} {x}" for d, t, x in iter_passages()}

qv = model.encode([f"query: {q}"], normalize_embeddings=True).astype("float32")
scores, idx = index.search(qv, 3)
for s, i in zip(scores[0], idx[0]):
    print(round(float(s), 3), docids[i], docid2text[docids[i]][:120])

0.898 35196#5 पेनिसिलिन पेनिसिलिन की खोज का श्रेय 1928 में स्कॉटिश वैज्ञानिक एवं नोबल पुरस्कार विजेता अलेक्ज़ेंडर फ्लेमिंग को जाता है।
0.898 35196#9 पेनिसिलिन 1939 में सर विलियम दुन स्कूल ऑफ पैथोलोजी, यूनिवर्सिटी ऑफ ऑक्सफोर्ड के ऑस्ट्रेलियाई वैज्ञानिक हॉवर्ड फ्लोरे (बा
0.89 47306#3 जैवप्रतिरोधी पेनिसिलिन की खोज एक प्रकार का संयोग (अप्रत्याशित घटना) था। सन 1928 में जबएलैग्जैंडर फ़्लेमिंग स्टैफिलोकोकस 


Save everything to Drive (separate folder from English)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/miracl_hi_indexes"
os.makedirs(DEST, exist_ok=True)

with open(f"{DEST}/passages.jsonl", "w", encoding="utf-8") as f:
    for docid, title, text in iter_passages():
        f.write(json.dumps({"id": docid, "title": title, "text": text}, ensure_ascii=False) + "\n")

import shutil
shutil.copytree(f"{WORK}/bm25_index", f"{DEST}/bm25_index", dirs_exist_ok=True)
shutil.copy(f"{WORK}/dense.faiss", DEST)
shutil.copy(f"{WORK}/docids.json", DEST)

for root, _, fs in os.walk(DEST):
    for fn in fs:
        p = os.path.join(root, fn)
        print(f"{os.path.getsize(p)/1e6:10.1f} MB  {os.path.relpath(p, DEST)}")

Mounted at /content/drive
     503.3 MB  passages.jsonl
     777.6 MB  dense.faiss
       6.1 MB  docids.json
       5.0 MB  bm25_index/_0_Lucene90_0.dvd
       0.0 MB  bm25_index/_0_Lucene104_0.tmd
       0.0 MB  bm25_index/segments_1
       0.0 MB  bm25_index/_0.fnm
       9.4 MB  bm25_index/_0_Lucene104_0.tim
       3.1 MB  bm25_index/_0.fdt
      28.3 MB  bm25_index/_0_Lucene104_0.doc
       0.0 MB  bm25_index/_0_Lucene90_0.dvm
       0.0 MB  bm25_index/_0_Lucene104_0.psm
       0.0 MB  bm25_index/_0_Lucene90_0.dvs
       0.2 MB  bm25_index/_0_Lucene104_0.tip
       0.0 MB  bm25_index/_0.fdm
       0.0 MB  bm25_index/_0.si
       0.5 MB  bm25_index/_0.nvd
       0.0 MB  bm25_index/_0.fdx
       0.0 MB  bm25_index/_0.nvm
       0.0 MB  bm25_index/write.lock
